In [8]:
library(dplyr)
library(tidyr)
library(purrr)
library(stringr)


source("NewSiteDataLoader.R")
source("data_loader.R")
source("meta.R")
source("recalMNE.R")
ls()

 [1] "accumulation_cause"                 "add_bone_info"                     
 [3] "add_element_count"                  "age_category"                      
 [5] "category_hierarchy"                 "category_mappings"                 
 [7] "category_mappings_arms_2_to_1"      "category_mappings_arms_3_to_1"     
 [9] "category_mappings_arms_3_to_2"      "category_mappings_foot_2_to_1"     
[11] "category_mappings_foot_3_to_1"      "category_mappings_foot_3_to_2"     
[13] "category_mappings_foot_4_to_1"      "category_mappings_foot_4_to_2"     
[15] "category_mappings_foot_4_to_3"      "category_mappings_foot_5_to_1"     
[17] "category_mappings_foot_5_to_2"      "category_mappings_foot_5_to_3"     
[19] "category_mappings_foot_5_to_4"      "category_mappings_hands_2_to_1"    
[21] "category_mappings_hands_3_to_1"     "category_mappings_hands_3_to_2"    
[23] "category_mappings_hands_4_to_1"     "category_mappings_hands_4_to_2"    
[25] "category_mappings_hands_4_to_3"     "category_

In [2]:
n_samples_small <- 1000 # 1k
n_samples_large <- 10000 # 10k
#n_samples_XL <- 500000 # 500k

HomininPartAbundance_small <- add_element_count(manipulate_rows(generate_dataset(n_samples_small)))
HomininPartAbundance_large <- add_element_count(manipulate_rows(generate_dataset(n_samples_large)))
#HomininPartAbundance_XL <- add_element_count(manipulate_rows(generate_dataset(n_samples_XL)))

# Save the datasets as .rd files
save(HomininPartAbundance_small, file = "HomininPartAbundance_small.rda")
save(HomininPartAbundance_large, file = "HomininPartAbundance_large.rda")
#save(HomininPartAbundance_XL, file = "HomininPartAbundance_XL.rda")


In [13]:
# Create NewSiteData_sum and NewSiteData_ind datasets
n_samples <- 1
NewSiteData_ind <- generate_dataset(n_samples)
NewSiteData_ind$number_of_individuals <- 24
NewSiteData_sum <- add_element_count(manipulate_rows(NewSiteData_ind))

NewSiteData_ind <- generate_dataset(n_samples)
NewSiteData_ind$number_of_individuals <- 7
NewSiteData_ind <- add_element_count(manipulate_rows(NewSiteData_ind))

# Save NewSiteData_sum and NewSiteData_ind as .rd files
save(NewSiteData_sum, file = "NewSiteData_sum.rda")
save(NewSiteData_ind, file = "NewSiteData_ind.rda")

print("### SUM ###")
print(colnames(NewSiteData_sum))

[1] "### SUM ###"
 [1] "site_index"            "sample_index"          "site_name"            
 [4] "sum"                   "time_range"            "geological_context"   
 [7] "number_of_individuals" "age"                   "gender"               
[10] "accumulation_cause"    "hominin_species"       "bone.level"           
[13] "element"               "count"                
[1] "### SUM ###"
 [1] "site_index"            "sample_index"          "site_name"            
 [4] "sum"                   "time_range"            "geological_context"   
 [7] "number_of_individuals" "age"                   "gender"               
[10] "accumulation_cause"    "hominin_species"       "bone.level"           
[13] "element"               "count"                


In [14]:
# Load the datasets
load("HomininPartAbundance_small.rda")
load("HomininPartAbundance_large.rda")
#load("HomininPartAbundance_XL.rda")
load("NewSiteData_sum.rda")
load("NewSiteData_ind.rda")

# If they are already tibbles, you don't need to convert them again.
# You can just start using them as they are
#print(HomininPartAbundance_small)
print("### SUM ###")
print(NewSiteData_sum)
print("### IND ###")
print(NewSiteData_ind)

[1] "### SUM ###"
# A tibble: 1 × 14
  site_index sample_index site_name   sum time_range         geological_context
       <int> <chr>        <chr>     <dbl> <chr>              <chr>             
1          1 11           Site_1        1 Late Modern Period Surface disposal  
# ℹ 8 more variables: number_of_individuals <dbl>, age <chr>, gender <chr>,
#   accumulation_cause <chr>, hominin_species <chr>, bone.level <list>,
#   element <list>, count <list>
[1] "### IND ###"
# A tibble: 7 × 14
  site_index sample_index site_name   sum time_range    geological_context
       <int> <chr>        <chr>     <dbl> <chr>         <chr>             
1          1 1_1          Site_1        0 Late Iron Age Cave              
2          1 1_2          Site_1        0 Late Iron Age Cave              
3          1 1_3          Site_1        0 Late Iron Age Cave              
4          1 1_4          Site_1        0 Late Iron Age Cave              
5          1 1_5          Site_1        0 Late Iron Age

In [15]:
result1 <- data.frame(
  bone.level = c(4,4,4,4,4,4, 2,3,3,4))
result2 <- data.frame(
  bone.level = c(1, 2, 3, 3, 4, 4, 4, 4, 4, 4))
result3 <- data.frame(
  bone.level = c(2,5,2,3,1,3, 1,2,3,4))

  min_bone_level <- compareBoneLevels(result1, result2, result3, NewSiteData_sum, NewSiteData_ind[1,],NewSiteData_ind[5,])
  print(min_bone_level)
print(unlist(NewSiteData_sum$bone.level, use.names = FALSE))

 [1] 1 1 1 1 1 2 1 1 1 3
 [1] 2 1 1 1 2 3 4 4 3 3
 [1] 1 1 1 1 1 2 1 1 1 3
 [1] 2 1 1 1 2 3 4 4 3 3


In [16]:
current_mne <- NewSiteData_sum$count[[1]]
bone_names <- NewSiteData_sum$element[[1]]
current_levels <- unlist(NewSiteData_sum$bone.level, use.names = FALSE)
target_levels <-  min_bone_level
sections_order <- c("Skull", "Hyoid", "Vertebrae", "Thorax", "Shoulder",
"Arms", "Hands", "Pelvic", "Legs", "Foot")

In [ ]:
NewSiteData_sum

In [ ]:
skeleton_tb <- get_bones_dataframe(target_levels)
sapply(skeleton_tb$count, generate_scaled_integer)

In [ ]:
print(generate_element_count(target_levels)$count)
print(get_bones_dataframe(target_levels)$count)

In [ ]:
source("meta.R")
library(dplyr)

for (i in 1:5) {
elements_data = elements_data
current_levels_n = unlist(generate_dataset(n_samples)$bone.level[[1]], use.names =FALSE)
input_df <- get_bones_dataframe(current_levels_n)
print(current_levels_n)
print(summarize_bone_sections(input_df, elements_data))
}

In [ ]:
source("meta.R")
library(dplyr)

elements_data = elements_data
#current_levels_n = unlist(generate_dataset(n_samples)$bone.level[[1]], use.names =FALSE)
input_df <- get_bones_dataframe(current_levels_n)
print(current_levels_n)
summarize_bone_sections(input_df, elements_data)

In [6]:
source("meta.R")
library(dplyr)

#for (i in 1:5) {
elements_data = elements_data
#current_levels_n = unlist(generate_dataset(n_samples)$bone.level[[1]], use.names =FALSE)
input_df <- get_bones_dataframe(current_levels)
#print(input_df$element)
cl_inp <- summarize_bone_sections(input_df, elements_data)
new_input_df <- get_bones_dataframe(cl_inp)$element
print(identical(input_df$element,new_input_df))
#print(input_df$element)
#print(new_input_df)
print(current_levels)
print(cl_inp)
# Convert both columns to character vectors (if they are not already)
col1 <- as.character(input_df$element)
col2 <- as.character(new_input_df)

# Check if the columns are identical
if (identical(col1, col2)) {
  print("The columns are identical.")
} else {
  print("The columns are not identical.")
  
  # Optionally, you can check for specific differences:
  print("Differences between the two columns:")
  differences <- which(col1 != col2)
  print(data.frame(Row = differences, input_df_element = col1[differences], get_bones_dataframe_element = col2[differences]))
}



[1] TRUE
 [1] 3 1 3 4 1 2 2 3 2 3
 [1] 3 1 3 4 1 2 2 3 2 3
[1] "The columns are identical."
[1] TRUE
 [1] 3 1 3 4 1 2 2 3 2 3
 [1] 3 1 3 4 1 2 2 3 2 3
[1] "The columns are identical."


In [ ]:
new_mne <- list()
current_tb <- add_bone_info(input_df)
for (part_index in seq_along(sections_order)) {
  part <- sections_order[part_index]
  current_level <- current_levels[[part_index]]
  target_level <- target_levels[[part_index]]

  if (current_level == target_level) {
    new_mne[[part]] <- (current_tb %>%
                         filter(Skeleton_Section == part))$count
    
      #print(new_mne)
    next
  }
  
mapping_name <- paste0(part, "_", current_level, "_to_", target_level)
print(mapping_name)

if (mapping_name %in% names(category_hierarchy)) {
  
  #mapping_function <- get(mapping_name, envir = category_hierarchy)
  aggregated_counts <- aggregate_counts_by_section(current_tb, part, current_level, target_level)
  print(aggregated_counts)

  if (part_index == 10){
    print("#####################")
    print("agg_counts")
    print(aggregated_counts)
    print("#####################")
  }
}

}



#print(current_levels)
#print(target_levels)
print(unname(unlist(new_mne)))
#print(current_tb$element)
print(current_tb$count)

In [ ]:
source("recalMNE.R")


aggregated_counts <- aggregate_counts_by_section(current_tb, part, current_level, target_level)
aggregated_counts


In [ ]:
source("meta.R")
element <- current_tb$element
#print(element)
category_mapping <-category_hierarchy[["Foot_2_to_1"]]
print(category_mapping)

for (category in category_mapping) {
  print(category_mapping)
  if (element %in% category_mapping) {


    return(category)
  }
}


In [ ]:
current_tb$element$

In [ ]:
print(current_levels)
print(target_levels)
category_hierarchy

In [ ]:
    #Skull 4 6 6
    #"Skull_4_to_2" = category_mappings_skull_4_to_2, # 5_3
    #"Skull_4_to_1" = category_mappings_skull_4_to_1, # 5_1
    #"Skull_3_to_2" = category_mappings_skull_3_to_2, # 4_3
    #"Skull_3_to_1" = category_mappings_skull_3_to_1, # 4_1
    #"Skull_2_to_1" = category_mappings_skull_2_to_1, # 3_1
   # "Skull_4_to_3" = category_mappings_skull_4_to_3, # 5_4
    "Skull_5_to_1" = category_mappings_hands_5_to_1, # 2_1
    "Skull_5_to_2" = category_mappings_hands_5_to_2, # 3_2
    "Skull_5_to_3" = category_mappings_hands_5_to_3, # 4_2
    "Skull_5_to_4" = category_mappings_hands_5_to_4, # 5_2

In [ ]:
skeleton_hierarchy <- list(
  Skull = list(
    "Level 1" = list("Skull" = 1),
    "Level 2" = list("Skull" = 1, "Mandible" = 1),
    "Level 3" = list(
      "Neurocranium bone" = 8,
      "Schplancnocranium" = 14,
      "Ossicles of the Middle Ear" = 6,
      "Mandible" = 1
    ),
    "Level 4" = list(
      Neurocranium_bone = c("Frontal Bone" = 1, "Parietal Bone" = 2, "Temporal" = 2, "Occipital" = 1, "Sphenoid" = 1, "Ethmoid" = 1),
      Schplancnocranium = c("Maxilla" = 2, "Palatine bone" = 2, "Zygomatic bone" = 2, "Nasal bone" = 2, "Lacrimal bone" = 2, "Vomer bone" = 1, "Inferior nasal concha" = 2),
      Ossicles_of_the_Middle_Ear = c("Malleus" = 2, "Incus" = 2, "Stapes" = 2),
      "Mandible" = 1
    ),
    "Level 5" = list(
      Neurocranium_bone = c("Frontal Bone" = 1, "Parietal Bone Right" = 1, "Parietal Bone Left" = 1, "Temporal Right" = 1, "Temporal Left" = 1, "Occipital" = 1, "Sphenoid" = 1, "Ethmoid" = 1),
      Schplancnocranium = c("Maxilla Right" = 1, "Maxilla Left" = 1, "Palatine bone Right" = 1, "Palatine bone Left" = 1, "Zygomatic bone Right" = 1, "Zygomatic bone Left" = 1, "Nasal bone Right" = 1, "Nasal bone Left" = 1, "Lacrimal bone Right" = 1, "Lacrimal bone Left" = 1, "Vomer bone" = 1, "Inferior nasal concha Right" = 1, "Inferior nasal concha Left" = 1),
      Ossicles_of_the_Middle_Ear = c("Malleus Right" = 1, "Malleus Left" = 1, "Incus Right" = 1, "Incus Left" = 1, "Stapes Right" = 1, "Stapes Left" = 1),
      "Mandible" = 1
    )
  ),

In [ ]:
   
   "Skull", "Neurocranium bone", "Frontal Bone", "Parietal Bone",
   
   
   #Skull
    c("Skull", "Neurocranium bone", "Frontal Bone", "Parietal Bone", "Parietal Bone Right",
    "Parietal Bone Left", "Temporal", "Temporal Right", "Temporal Left", "Occipital",
    "Sphenoid", "Ethmoid", "Schplancnocranium", "Maxilla", "Palatine bone",
    "Zygomatic bone", "Zygomatic bone Right", "Zygomatic bone Left", "Nasal bone",
    "Lacrimal bone", "Vomer bone", "Inferior nasal concha", "Ossicles of the Middle Ear",
    "Malleus", "Malleus Right", "Malleus Left", "Incus", "Incus Right", "Incus Left",
    "Stapes", "Stapes Right", "Stapes Left", "Mandible")

In [11]:
recal_mne <- function(current_mne, current_levels, target_levels, skeleton_hierarchy, bone_names, sections_order) {
  
  # Initialize an empty list to store new MNE results
  new_mne <- list()
  
  # Create the initial tibble with counts and elements
  current_mne_named <- tibble(count = current_mne, element = bone_names)
  
  # Add bone info to the tibble (assuming add_bone_info function exists)
  current_mne_named <- add_bone_info(current_mne_named)
  
  # Helper function to standardize names
  standardize_name <- function(name) {
    gsub(" ", ".", name)
  }
  
  # Loop over each part/section
  for (part_index in seq_along(sections_order)) {
    part <- sections_order[part_index]
    current_level <- current_levels[[part_index]]
    target_level <- target_levels[[part_index]]
    
    # If current level matches target level, no aggregation is needed; just copy the original counts
    if (current_level == target_level) {
      # Extract counts for this part and add directly to new_mne
      part_data <- current_mne_named %>% filter(Skeleton_Section == part)
      for (i in seq_len(nrow(part_data))) {
        element <- standardize_name(part_data$element[i])
        count <- part_data$count[i]
        new_mne[[paste(part, element, sep = "_")]] <- count
      }
      next
    }
    
    # Create the appropriate mapping name for this part and level transition
    mapping_name <- paste0(part, "_", current_level, "_to_", target_level)
    
    # Check if the mapping exists in the hierarchy
    if (mapping_name %in% names(category_hierarchy)) {
      # Get the mapping
      mapping <- category_hierarchy[[mapping_name]]
      
      # Build a lookup from elements to target categories
      element_to_category <- unlist(lapply(names(mapping), function(category) {
        std_category <- standardize_name(category)
        elements <- mapping[[category]]
        std_elements <- sapply(elements, standardize_name)
        setNames(rep(std_category, length(std_elements)), std_elements)
      }))
      
      # Filter data for this part
      part_data <- current_mne_named %>% filter(Skeleton_Section == part)
      
      # Standardize element names
      part_data$std_element <- standardize_name(part_data$element)
      
      # Map elements to target categories
      part_data$target_category <- element_to_category[part_data$std_element]
      
      # Handle elements not in mapping (optional)
      # For now, exclude them
      part_data <- part_data %>% filter(!is.na(target_category))
      
      # Aggregate counts by target category
      aggregated_counts <- part_data %>%
        group_by(target_category) %>%
        summarize(total_count = sum(count, na.rm = TRUE))

      aggregated_counts$target_category <- factor( aggregated_counts$target_category, levels = names(mapping))
      aggregated_counts <- aggregated_counts %>%
      arrange(target_category)
      
      # Store the counts in new_mne
      for (i in seq_len(nrow(aggregated_counts))) {
        category <- aggregated_counts$target_category[i]
        total_count <- aggregated_counts$total_count[i]
        new_mne[[paste(part, category, sep = "_")]] <- total_count
      }
    } else {
      # Mapping not found, handle accordingly
      warning(paste("Mapping not found for", mapping_name))
    }
  }
  
  # Convert new_mne to data frame
  new_mne_df <- tibble(element = names(new_mne), count = unlist(new_mne))
  
  # Return the new MNE results
  return(new_mne_df)
}


In [12]:
source("meta.R")
# Assuming you have defined the necessary variables:
# current_mne, current_levels, target_levels, skeleton_hierarchy, bone_names, sections_order, category_hierarchy

# Call the function
new_mne_results <- recal_mne(
  current_mne = current_mne,
  current_levels = current_levels,
  target_levels = target_levels,
  skeleton_hierarchy = skeleton_hierarchy,
  bone_names = bone_names,
  sections_order = sections_order
)

# View the recalculated MNE counts
print(current_levels)
print(target_levels)
print(new_mne_results, n=25)


 [1] 3 1 3 4 1 2 2 3 2 3
 [1] 1 1 1 3 1 2 1 2 1 1
# A tibble: 15 × 2
   element             count
   <chr>               <dbl>
 1 Skull_Skull            17
 2 Hyoid_Hyoid             0
 3 Vertebrae_Vertebrae    12
 4 Thorax_Sternum          1
 5 Thorax_Ribs            10
 6 Shoulder_Shoulder       3
 7 Arms_Humerus            1
 8 Arms_Radius             0
 9 Arms_Ulna               1
10 Hands_Hands            33
11 Pelvic_Sacrum           4
12 Pelvic_Coccyx           2
13 Pelvic_Os.Coxae         3
14 Legs_Legs               2
15 Foot_Foot              13
 [1] 3 1 3 4 1 2 2 3 2 3
 [1] 1 1 1 3 1 2 1 2 1 1
# A tibble: 15 × 2
   element             count
   <chr>               <dbl>
 1 Skull_Skull            17
 2 Hyoid_Hyoid             0
 3 Vertebrae_Vertebrae    12
 4 Thorax_Sternum          1
 5 Thorax_Ribs            10
 6 Shoulder_Shoulder       3
 7 Arms_Humerus            1
 8 Arms_Radius             0
 9 Arms_Ulna               1
10 Hands_Hands            33
11 Pelvic_Sacrum     

In [ ]:
for (part_index in 10:10){
  part <- sections_order[part_index]
  current_mne_named %>% filter(Skeleton_Section == part) -> part_data
  print(sum(part_data$count))
  print(part_data, n=25)   

}


In [ ]:
sum(current_mne)

In [ ]:
sum(new_mne_results$count)

In [9]:
print(current_levels)
print(target_levels)
 # Initialize an empty list to store new MNE results
  new_mne <- list()
  
  # Create the initial tibble with counts and elements
  current_mne_named <- tibble(count = current_mne, element = bone_names)
  
  # Add bone info to the tibble (assuming add_bone_info function exists)
  current_mne_named <- add_bone_info(current_mne_named)
  #print(current_mne_named$Level)
  # Helper function to standardize names
  standardize_name <- function(name) {
    gsub(" ", ".", name)
  }
  
  # Loop over each part/section
  for (part_index in 10:10) {
    part <- sections_order[part_index]
    current_level <- current_levels[[part_index]]
    target_level <- target_levels[[part_index]]
    
    # If current level matches target level, no aggregation is needed; just copy the original counts
    if (current_level == target_level) {
      # Extract counts for this part and add directly to new_mne
      part_data <- current_mne_named %>% filter(Skeleton_Section == part)
      for (i in seq_len(nrow(part_data))) {
        element <- standardize_name(part_data$element[i])
        count <- part_data$count[i]
        new_mne[[paste(part, element, sep = "_")]] <- count
      }
      next
    }
    
    # Create the appropriate mapping name for this part and level transition
    mapping_name <- paste0(part, "_", current_level, "_to_", target_level)
    
    if (mapping_name %in% names(category_hierarchy)) {
      # Get the mapping
      mapping <- category_hierarchy[[mapping_name]]
      
      # Build a lookup from elements to target categories
      element_to_category <- unlist(lapply(names(mapping), function(category) {
        std_category <- standardize_name(category)
        elements <- mapping[[category]]
        std_elements <- sapply(elements, standardize_name)
        setNames(rep(std_category, length(std_elements)), std_elements)
      }))


      # Filter data for this part
      part_data <- current_mne_named %>% filter(Skeleton_Section == part)
      
      # Standardize element names
      part_data$std_element <- standardize_name(part_data$element) #not needed
      
      # Map elements to target categories
      part_data$target_category <- element_to_category[part_data$std_element] #not needed

      # After the aggregation process
      aggregated_counts <- part_data %>%
        group_by(target_category) %>%
        summarize(total_count = sum(count, na.rm = TRUE), .groups = 'drop')

      # Reorder target_category based on the names in mapping
      aggregated_counts$target_category <- factor(aggregated_counts$target_category, levels = names(mapping))

      aggregated_counts <- aggregated_counts %>%
            arrange(target_category)

      # Now it should display in the desired order
      print(aggregated_counts)

      print(rownames(mapping))
      print(part_data$target_category)

    }
    
  }

  print(part_data%>%  group_by(target_category) %>%  summarize(total_count = sum(count, na.rm = TRUE),.groups = 'drop') %>% arrange(match(target_category,mapping)) , n= 45)

 [1] 3 1 3 4 1 2 2 3 2 3
 [1] 1 1 1 3 1 2 1 2 1 1
# A tibble: 1 × 2
  target_category total_count
  <fct>                 <dbl>
1 Foot                     13
NULL
    Foot.Tarsal.Right      Foot.Tarsal.Left Foot.Metatarsal.Right 
               "Foot"                "Foot"                "Foot" 
 Foot.Metatarsal.Left    Foot.Phalanx.Right     Foot.Phalanx.Left 
               "Foot"                "Foot"                "Foot" 
# A tibble: 1 × 2
  target_category total_count
 [1] 3 1 3 4 1 2 2 3 2 3
 [1] 1 1 1 3 1 2 1 2 1 1
# A tibble: 1 × 2
  target_category total_count
  <fct>                 <dbl>
1 Foot                     13
NULL
    Foot.Tarsal.Right      Foot.Tarsal.Left Foot.Metatarsal.Right 
               "Foot"                "Foot"                "Foot" 
 Foot.Metatarsal.Left    Foot.Phalanx.Right     Foot.Phalanx.Left 
               "Foot"                "Foot"                "Foot" 
# A tibble: 1 × 2
  target_category total_count


  <chr>                 <dbl>
1 Foot                     13
  <chr>                 <dbl>
1 Foot                     13


In [ ]:
part_data$std_element

In [ ]:
aggregated_counts$target_category

In [ ]:
print(names(mapping))

In [ ]:
target_category

In [10]:
aggregated_counts$target_category <- factor( aggregated_counts$target_category, levels = names(mapping))
aggregated_counts <- aggregated_counts %>%
arrange(target_category)

aggregated_counts

# A tibble: 1 × 2
  target_category total_count
  <fct>                 <dbl>
1 Foot                     13